In [32]:
import os
import pandas as pd
from collections import defaultdict
import tqdm

# 1.–– Paths
current_dir       = os.getcwd()
parquet_dir       = os.path.join(current_dir, "ParquetFilesWindows")
parquet_files     = os.listdir(parquet_dir)

daily_series_path = (
    r"C:/Users/ellis/OneDrive - Newcastle University/"
    "SESDP/Final Rainfall Datasets/WETTERtimeseries.csv"
)

# 2.–– Read your daily series WITH day-first parsing
daily_series = pd.read_csv(
    daily_series_path,
    index_col="Date",
    parse_dates=True,
    dayfirst=True            # ← crucial for “13/01/2025”
)
# Now daily_series.index is a DatetimeIndex (at midnight)

# 3.–– Prepare accumulators
overflow_flag = defaultdict(bool)
depth_sum     = defaultdict(float)
depth_count   = defaultdict(int)

# 4.–– Stream through each parquet

for fname in tqdm.tqdm(parquet_files):
    fpath = os.path.join(parquet_dir, fname)
    df = pd.read_parquet(fpath, columns=["Depth", "Overflow"])
    
    # ensure datetime index, extract DATE
    df.index = pd.to_datetime(df.index)
    df["Date"] = df.index.date
    
    daily = (
        df.groupby("Date")
          .agg(
             any_overflow = ("Overflow", lambda x: x.gt(0).any()),
             depth_sum    = ("Depth",   "sum"),
             depth_count  = ("Depth",   "count")
          )
    )
    
    # accumulate
    for date, row in daily.iterrows():
        overflow_flag[date] = overflow_flag[date] or row["any_overflow"]
        depth_sum[date]    += row["depth_sum"]
        depth_count[date]  += row["depth_count"]

    count += 1

# 5.–– Build metrics DataFrame
# Convert python-date keys → Timestamps at midnight
dates = pd.to_datetime(list(depth_sum.keys()))
metrics = pd.DataFrame({
    "OverflowOccurred": [overflow_flag[d.date()] for d in dates],
    "AvgDepth":         [depth_sum[d.date()]/depth_count[d.date()] for d in dates]
}, index=dates)
metrics.index.name = "Date"

# 6.–– Single join on DatetimeIndex
daily_series = daily_series.join(metrics, how="left")

print("✓ Done: daily_series now has exactly two new columns.")
daily_series


100%|██████████| 1826/1826 [10:49<00:00,  2.81it/s]

✓ Done: daily_series now has exactly two new columns.


,Amount,OverflowOccurred,AvgDepth
Date,,,
2025-01-01,0.066781,False,0.076046
2025-01-02,2.609644,False,0.994884
2025-01-03,0.378507,False,1.000227
2025-01-04,23.842416,True,1.005405
2025-01-05,2.364699,False,1.001616
...,...,...,...
2099-12-27,0.901811,False,1.000731
2099-12-28,0.225275,False,1.000003
2099-12-29,0.000000,NaN,NaN


In [34]:
pct_overflow_nonzero = daily_series.loc[daily_series['Amount'] != 0, 'OverflowOccurred'].mean() * 100
print(daily_series["AvgDepth"].mean(),"\n",100-pct_overflow_nonzero)

0.9983686027101324 
 81.52361977664762
